# conv-leakyrelu-block-discriminator — worked example 2: First DCGAN discriminator block without BatchNorm

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-leakyrelu-block-discriminator`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a real DCGAN discriminator the FIRST block omits BatchNorm — it consumes the raw image whose statistics should not be normalized away. So the first block is just `Conv2d(kernel=4, stride=2, padding=1, bias=True) -> LeakyReLU(0.2)`. Because there is no following BatchNorm, the conv keeps its bias term.

## Worked solution

**Step 1 — why no BatchNorm.** The first layer sees the raw input image (e.g. 3 RGB channels). Normalizing those statistics with BatchNorm removes information about absolute pixel scale that the discriminator may want; the DCGAN paper therefore drops BN on the first block.

**Step 2 — conv with bias.** Since no BatchNorm follows, we set `bias=True` (the default). The conv is still `kernel_size=4, stride=2, padding=1`, halving spatial size: `floor((32+2-4)/2)+1 = 16`.

**Step 3 — activation.** `nn.LeakyReLU(0.2, inplace=True)` follows directly, with no BatchNorm in between.

**Step 4 — verify.** A `(2, 3, 32, 32)` image becomes `(2, 64, 16, 16)`, and crucially `block[0].bias` is NOT None because we kept the bias.

In [ ]:
import torch.nn as nn

def build_first_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=True),
        nn.LeakyReLU(0.2, inplace=True),
    )

t.manual_seed(0)
block = build_first_block(3, 64)
x = t.randn(2, 3, 32, 32)
out = block(x)
print(tuple(out.shape))
print(block[0].bias is not None)
print(len(block))